In [1]:
import pandas as pd

In [7]:
orders = pd.read_csv("Orders.csv" , sep=';')
print(orders.head())
print(orders.columns)

   Row ID        Order ID  Order Date   Ship Date       Ship Mode Customer ID  \
0       1  CA-2022-152156  08/11/2022  11/11/2022    Second Class    CG-12520   
1       2  CA-2022-152156  08/11/2022  11/11/2022    Second Class    CG-12520   
2       3  CA-2022-138688  12/06/2022  16/06/2022    Second Class    DV-13045   
3       4  US-2021-108966  11/10/2021  18/10/2021  Standard Class    SO-20335   
4       5  US-2021-108966  11/10/2021  18/10/2021  Standard Class    SO-20335   

     Segment  Postal Code       Product ID     Sales  Quantity Discount  \
0   Consumer        42420  FUR-BO-10001798    261,96         2        0   
1   Consumer        42420  FUR-CH-10000454    731,94         3        0   
2  Corporate        90036  OFF-LA-10000240     14,62         2        0   
3   Consumer        33311  FUR-TA-10000577  957,5775         5     0,45   
4   Consumer        33311  OFF-ST-10000760    22,368         2      0,2   

     Profit  
0   41,9136  
1   219,582  
2    6,8714  
3  -38

In [9]:
# Fix numeric columns
orders["Sales"] = orders["Sales"].str.replace(",", ".", regex=False).astype(float)
orders["Profit"] = orders["Profit"].str.replace(",", ".", regex=False).astype(float)

# Quick validation
print(orders[["Sales", "Profit"]].head())
print(orders[["Sales", "Profit"]].dtypes)

      Sales    Profit
0  261.9600   41.9136
1  731.9400  219.5820
2   14.6200    6.8714
3  957.5775 -383.0310
4   22.3680    2.5164
Sales     float64
Profit    float64
dtype: object


In [11]:
# Aggregate to customer level
customer_df = orders.groupby("Customer ID").agg(
    Total_Sales=("Sales", "sum"),
    Total_Profit=("Profit", "sum"),
    Orders=("Order ID", "count"),
    Avg_Order_Value=("Sales", "mean")
).reset_index()

# Validation
print(customer_df.head())
print("\nSummary stats:")
print(customer_df.describe())

  Customer ID  Total_Sales  Total_Profit  Orders  Avg_Order_Value
0    AA-10315     5563.560     -362.8825      11       505.778182
1    AA-10375     1056.390      277.3824      15        70.426000
2    AA-10480     1790.512      435.8274      12       149.209333
3    AA-10645     5086.935      857.8033      18       282.607500
4    AB-10015      886.156      129.3465       6       147.692667

Summary stats:
        Total_Sales  Total_Profit      Orders  Avg_Order_Value
count    793.000000    793.000000  793.000000       793.000000
mean    2896.848500    361.156396   12.602774       227.868165
std     2628.670117    894.261812    6.242559       190.342560
min        4.833000  -6626.389500    1.000000         2.416500
25%     1146.050000     36.613100    8.000000       115.520200
50%     2256.394000    227.833800   12.000000       183.924000
75%     3785.276000    560.007800   16.000000       282.688947
max    25043.050000   8981.323900   37.000000      1751.292000


In [13]:
from sklearn.preprocessing import StandardScaler

# Features for clustering
features = ["Total_Sales", "Total_Profit", "Orders", "Avg_Order_Value"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(customer_df[features])

# Validation
print(X_scaled[:5])

[[ 1.01511197 -0.81016078 -0.25691161  1.46097336]
 [-0.70059002 -0.09373862  0.38425569 -0.82767369]
 [-0.4211387   0.08355286 -0.09661979 -0.41350959]
 [ 0.83367962  0.55572131  0.86513117  0.28776477]
 [-0.76539139 -0.25938287 -1.05837074 -0.42148271]]


In [15]:
from sklearn.cluster import KMeans

# Apply K-Means clustering
kmeans = KMeans(n_clusters=3, random_state=42)
customer_df["Cluster"] = kmeans.fit_predict(X_scaled)

# Validation
print(customer_df["Cluster"].value_counts())


Cluster
1    456
0    283
2     54
Name: count, dtype: int64


In [17]:
# Inspect cluster characteristics
cluster_summary = customer_df.groupby("Cluster").agg(
    Avg_Total_Sales=("Total_Sales", "mean"),
    Avg_Total_Profit=("Total_Profit", "mean"),
    Avg_Orders=("Orders", "mean"),
    Avg_Order_Value=("Avg_Order_Value", "mean"),
    Customers=("Customer ID", "count")
)

print(cluster_summary)

         Avg_Total_Sales  Avg_Total_Profit  Avg_Orders  Avg_Order_Value  \
Cluster                                                                   
0            3957.208378        472.524701   18.151943       230.575006   
1            1461.818839         90.119864    8.765351       171.759201   
2            9457.805526       2066.256915   15.925926       687.491334   

         Customers  
Cluster             
0              283  
1              456  
2               54  


In [19]:
segment_map = {
    0: "Medium Value",
    1: "Low Value",
    2: "High Value"
}

customer_df["Segment"] = customer_df["Cluster"].map(segment_map)


In [21]:
print(customer_df[["Customer ID", "Segment"]].head())
print(customer_df["Segment"].value_counts())


  Customer ID       Segment
0    AA-10315  Medium Value
1    AA-10375     Low Value
2    AA-10480     Low Value
3    AA-10645  Medium Value
4    AB-10015     Low Value
Segment
Low Value       456
Medium Value    283
High Value       54
Name: count, dtype: int64


In [23]:
customer_df.to_csv("customer_segments.csv", index=False)
print("customer_segments.csv created successfully")


customer_segments.csv created successfully
